# AVCS 2.0M Scaled Cortical Visual Foundation Model — Kaggle GPU Training

**Target Architecture**: `UnifiedCortex2M` (~2.40 Million Parameters)
**Objective**: End-to-end multi-task training on GPU (T4 / P100 / Dual-T4) with zero compromise:
1. **24-Class Spatial Affordance Heatmaps** (Buttons, Inputs, Code Lines, Cells, Tabs, Handles)
2. **32 Universal Desktop Application Categories** (IDE, Terminal, CAD, Canvas, Office, Media, OS)
3. **Zero-Shot Natural Language Gaze Grounding** (Linguistic Cross-Attention Modulation)
4. **256-Dimensional Metric Visual Embeddings** for Persistent Element Re-ID across scrolling

### Kaggle Setup Instructions:
- **Accelerator**: Select **GPU T4 x2** or **GPU P100** under Notebook options (Right sidebar).
- **Internet**: Toggle **Internet ON** under Notebook options.

In [ ]:
# Step 1: Check GPU Environment and Accelerators
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Count    : {torch.cuda.device_count()}")
else:
    print("WARNING: Running on CPU. Please enable GPU accelerator in Kaggle sidebar.")

In [ ]:
# Step 2: Clone or Pull Latest AVCS Repository
!mkdir -p /kaggle/working/packages
!if [ -d /kaggle/working/packages/avcs/.git ]; then git -C /kaggle/working/packages/avcs pull origin main; else git clone https://github.com/Parth-labs-in/AVCS.git /kaggle/working/packages/avcs; fi
%cd /kaggle/working/packages/avcs

In [ ]:
# Step 3: Install Core Dependencies
!pip install numpy torch --quiet

In [ ]:
# Step 4: Launch Full Production 2.0M Model Training on GPU
# 50,000 screen interaction pairs across all 32 desktop app categories and 24 affordance geometries
!python avcs/training/train_kaggle.py \
    --epochs 20 \
    --batch-size 64 \
    --lr 5e-4 \
    --samples 50000 \
    --val-samples 5000 \
    --output-dir /kaggle/working/runs/cortex_2m

In [ ]:
# Step 5: Verify Saved Production Checkpoint
import json
from pathlib import Path

ckpt_path = Path("/kaggle/working/runs/cortex_2m/cortex_2m_best.pt")
assert ckpt_path.exists(), "Training checkpoint not found!"

ckpt = torch.load(ckpt_path, map_location="cpu")
print("=" * 70)
print("CHECKPOINT CERTIFICATION REPORT:")
print(f"  • Best Epoch           : {ckpt['epoch']}")
print(f"  • Best Validation Loss : {ckpt['val_loss']:.4f}")
print(f"  • Category Accuracy    : {ckpt['val_cat_acc']:.2f}%")
print(f"  • Total Parameters     : {ckpt['total_parameters']:,} (~2.40M)")
print(f"  • Checkpoint File Size : {ckpt_path.stat().st_size / (1024*1024):.2f} MB")
print("=" * 70)

# Show Training History
history_file = Path("/kaggle/working/runs/cortex_2m/training_history.json")
if history_file.exists():
    with open(history_file) as f:
        meta = json.load(f)
    print(f"Total GPU Training Duration: {meta['total_training_seconds'] / 60.0:.2f} minutes")

In [ ]:
# Step 6: Test Inference with the Trained 2.40M Model
import sys
sys.path.insert(0, '/kaggle/working/packages/avcs')

from avcs.cortex_2m import HumanEye2M, UnifiedCortex2M
from forge.vision.capture import SyntheticScreen

cortex = UnifiedCortex2M()
cortex.load_state_dict(ckpt['model_state_dict'])
cortex.eval()

eye = HumanEye2M(source=SyntheticScreen())
eye.cortex = cortex

res = eye.perceive()
print("Live Inference Result:")
print(f"  • Detected App Category: {res['primary_category']} (Conf: {res['category_confidence']*100:.1f}%)")
print(f"  • Affordance Hotspots  : {res['affordances']}")
print(f"  • Latency              : {res['latency_ms']:.2f} ms")
eye.close()